In [1]:
import os
import pandas as pd
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-07 21:29:14.212459


### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

Project: 20231010-gen-xii
Task: 08_retro_scoring
Subtask: 06_create_df


### Get the production data

In [4]:
%%time

str_filename = 'df_X_raw.gzip'
str_uri = f's3://{str_project}/{str_task}/04_concat_results/{str_filename}'
df_X_raw = pd.read_parquet(str_uri)

# convert to dtm
df_X_raw['dtmFunded'] = pd.to_datetime(df_X_raw['dtmFunded'])
# sort
df_X_raw.sort_values(by='dtmFunded', ascending=True, inplace=True)

# list cols to drop because they are already in tsp
list_cols = [
    'bigaccountid__app',
    'dtmfunded__app',
    'bigdebtorid__app',
    'bitdebtor__app',
    'dtmstampcreation__app',
    'intterm__app',
    'fltdowncash__app',
    'fltapproveddowntotal__app',
    'bitservicecontract__app',
    'applicationdate__app',
    
    'monthonbooks__app',
    'runningnetloss__app',
    'amtfinanced__app',
    'intopenbktype__app',
    'vehicleyear__app',
    'bitnew__app',
    'vehiclemake__app',
    'miles_odometer__app',
    'bookvalue__app',
    'payment__app',
    'dti__app',
    'pti__app',
    'fltadvance__app',
    'strvehicletype__app',
    'bitgap__app',
    'dealerstampcreation__app',
]
df_X_raw.drop(list_cols, axis=1, inplace=True)

# show
df_X_raw

CPU times: user 5.65 s, sys: 2.87 s, total: 8.52 s
Wall time: 3.49 s


,uniqueid__app,bigstatusid__app,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,approvaldate__app,bitfunded__app,fundeddate__app,...,score_bankcard__tu,score_cvpropensity__tu,finscore__tu,intscore__ln,target,bigAccountId,dtmFunded,BITDEBTOR,ln_was_empty__ln,tu_was_empty__tu
5514485__primary__20210120,0__0__20210120,14.0,Stanley,Virginia,22851,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,5514485,2021-01-25,1,NaN,NaN
5515245__primary__20210120,0__0__20210120,14.0,Austin,Texas,78727,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,5515245,2021-01-27,1,NaN,NaN
5517730__primary__20210122,0__0__20210122,14.0,STOCKTON,California,95207,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,5517730,2021-01-27,1,NaN,NaN
5518455__primary__20210123,0__0__20210123,14.0,Mount Orab,Ohio,45154,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,5518455,2021-01-28,1,NaN,NaN
5518455__secondary__20210123,0__0__20210123,14.0,Mount Orab,Ohio,45154,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,5518455,2021-01-28,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7321766__primary__20231109,0__0__20231109,5.0,FOREST HILL,Texas,76119,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,7321766,2024-02-08,1,NaN,NaN
7359135__primary__20231120,0__0__20231120,5.0,CHICAGO,Illinois,60629,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,7359135,2024-02-09,1,NaN,NaN
7306479__primary__20231107,0__0__20231107,1.0,PIKESVILLE,Maryland,21208,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,7306479,2024-02-09,1,NaN,NaN
7351735__primary__20231117,0__0__20231117,5.0,Saginaw,Michigan,48601,False,0.0,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,7351735,2024-02-28,1,NaN,NaN


### Get the data from TSP

In [5]:
%%time

str_filename = 'df_scored_accounts.csv'
str_uri = f's3://{str_project}/{str_task}/05_get_tsp_from_db/{str_filename}'
df_tsp = pd.read_csv(str_uri)

# convert to dtm
df_tsp['dtmstampcreation__app'] = pd.to_datetime(df_tsp['dtmstampcreation__app'])
df_tsp['dtmfunded__app'] = pd.to_datetime(df_tsp['dtmfunded__app'])
# sort
df_tsp.sort_values(by='dtmfunded__app', ascending=True, inplace=True)
# drop the oldest repeated unique id
df_tsp.drop_duplicates(subset=['uniqueid__app'], keep='last', inplace=True)

# rename
dict_rename = {
    'dtmstampcreation__app': 'applicationdate__app',
}
df_tsp.rename(columns=dict_rename, inplace=True)

# show
df_tsp

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:273: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 635 ms, sys: 47.2 ms, total: 683 ms
Wall time: 1.65 s


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,applicationdate__app,dtmfunded__app,bitdefault__app,monthonbooks__app,runningnetloss__app,amtfinanced__app,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
188,546268468806661,5462684,6880666,1,2020-12-28 18:42:52.023,2021-01-11,1,38.0,2508.84,18684.50,...,0.0,0.0,498.00,0.288049,0.102610,0,1.149965,auto,0,2019-08-14 14:22:39.430
26,545783068747971,5457830,6874797,1,2020-12-21 16:59:56.593,2021-01-11,1,38.0,9361.83,12906.00,...,500.0,500.0,361.96,0.298214,0.143694,0,1.163820,suv,0,2018-02-22 17:06:05.777
142,546012068775641,5460120,6877564,1,2020-12-26 07:05:54.407,2021-01-11,0,38.0,0.00,28998.67,...,0.0,0.0,674.54,0.346866,0.103493,1,0.936217,suv,1,2014-07-30 15:56:47.253
69719,545482668711180,5454826,6871119,0,2020-12-18 10:50:31.303,2021-01-11,0,38.0,0.00,28721.72,...,0.0,0.0,594.95,0.377412,0.101147,1,1.124077,auto,1,2014-03-04 14:26:14.460
133,546791868870881,5467918,6887088,1,2021-01-04 09:34:10.250,2021-01-11,0,38.0,0.00,21123.60,...,0.0,0.0,459.11,0.494103,0.104108,0,1.149467,auto,1,2013-02-18 16:07:17.973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51740,759611194316541,7596111,9431654,1,2024-02-22 16:49:00.020,2024-03-06,0,NaN,NaN,23279.58,...,2500.0,2000.0,662.12,NaN,0.111901,1,0.891665,suv,1,2020-01-16 11:47:35.830
51741,759636494319601,7596364,9431960,1,2024-02-22 17:31:42.417,2024-03-06,0,NaN,NaN,15693.61,...,4000.0,5000.0,443.98,NaN,0.106555,0,1.137805,suv,0,2024-01-23 16:32:42.323
51748,759754594333701,7597545,9433370,1,2024-02-23 09:42:34.740,2024-03-06,0,NaN,NaN,19343.43,...,1250.0,1000.0,532.99,NaN,0.152149,0,1.077742,suv,0,2022-11-18 16:42:42.383
51848,760344794406021,7603447,9440602,1,2024-02-24 11:15:10.620,2024-03-06,0,NaN,NaN,23772.73,...,1000.0,1000.0,711.55,NaN,0.066921,0,1.149933,auto,0,2018-02-16 12:20:23.657


### Join

In [6]:
%%time

df = pd.merge(
    left=df_X_raw,
    right=df_tsp,
    left_on=['bigAccountId','BITDEBTOR'],
    right_on=['bigaccountid__app','bitdebtor__app'],
    how='inner',
)

# show
df

CPU times: user 282 ms, sys: 244 ms, total: 526 ms
Wall time: 525 ms


,uniqueid__app_x,bigstatusid__app,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,approvaldate__app,bitfunded__app,fundeddate__app,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
0,0__0__20210120,14.0,Stanley,Virginia,22851,False,0.0,NaN,False,NaN,...,0.0,0.00,412.34,0.307483,0.103700,0,1.149789,suv,1,2011-06-30 10:15:42.320
1,0__0__20210120,14.0,Austin,Texas,78727,False,0.0,NaN,False,NaN,...,250.0,105.06,638.00,0.233488,0.132294,0,1.100576,suv,0,2012-10-19 15:20:30.163
2,0__0__20210122,14.0,STOCKTON,California,95207,False,0.0,NaN,False,NaN,...,1000.0,1000.00,629.99,0.340494,0.140493,0,1.051327,suv,1,2020-07-06 17:09:57.383
3,0__0__20210123,14.0,Mount Orab,Ohio,45154,False,0.0,NaN,False,NaN,...,0.0,0.00,398.95,0.481666,0.070066,0,1.143705,auto,0,2016-11-16 16:28:03.170
4,0__0__20210123,14.0,Mount Orab,Ohio,45154,False,0.0,NaN,False,NaN,...,0.0,0.00,398.95,0.481666,0.070066,0,1.143705,auto,0,2016-11-16 16:28:03.170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71184,0__0__20231109,5.0,FOREST HILL,Texas,76119,False,0.0,NaN,False,NaN,...,0.0,0.00,511.77,0.449811,0.122139,0,1.349948,suv,0,2008-02-20 11:14:02.013
71185,0__0__20231120,5.0,CHICAGO,Illinois,60629,False,0.0,NaN,False,NaN,...,1000.0,500.00,450.63,0.350022,0.150022,0,1.149932,suv,0,2013-05-08 08:47:59.997
71186,0__0__20231107,1.0,PIKESVILLE,Maryland,21208,False,0.0,NaN,False,NaN,...,500.0,1000.00,727.74,0.385896,0.058485,1,0.838573,suv,0,2023-05-03 15:53:03.583
71187,0__0__20231117,5.0,Saginaw,Michigan,48601,False,0.0,NaN,False,NaN,...,0.0,0.00,587.21,0.453596,0.128596,0,1.121384,suv,1,2017-07-26 16:22:04.930


### Write to s3

In [7]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 24.4 s, sys: 146 ms, total: 24.6 s
Wall time: 24.6 s
